<a href="https://colab.research.google.com/github/Anagh19/Complex-Systems-Project/blob/all-states-adding/CS7065_DOI_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Setting up the import from Github for update ease, only need to run once per session

In [242]:
from pathlib import Path

repo_path = Path("/content/Complex-Systems-Project")

if not repo_path.exists():
    !git clone https://github.com/Anagh19/Complex-Systems-Project

%cd /content/Complex-Systems-Project

PROJECT_ROOT = Path.cwd()

/content/Complex-Systems-Project


Package Imports

In [264]:
try:
  import mesa
except:
  !pip install mesa --quiet
  import mesa
try:
  import networkx as nx
except:
  !pip install networkx --quiet
  import networkx as nx
try:
  import geopandas as gpd
except:
  !pip install geopandas --quiet
  import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import math
import random
from shapely.geometry import Point, Polygon, MultiPolygon
import plotly.express as px
from shapely.ops import unary_union
import seaborn as sns

Creating State Shapes and Other GeoJSON Formatting Things

In [244]:
gdf = gpd.read_file('All_States.geojson')
# Dissolve county polygons into state polygons and set STATEFP as index
state_gdf = gdf.dissolve(by='STATEFP', aggfunc='first')

# Remove Alaska (02), Hawaii (15), and Puerto Rico (72) sorry
states_to_remove = ['02', '15', '72']
state_gdf = state_gdf[~state_gdf.index.isin(states_to_remove)]

STATE_IDS = state_gdf.index.unique()
STATE_POLYGONS = state_gdf.geometry

HEALTH AGENT DEVELOPMENT!! \\
still need to develop spreading actions according to rules and maybe add color coding

In [255]:
class Health_Agent(mesa.Agent):
  def __init__(
      self,
      model,
      agent_type, # LOCAL or STATE
      state_id,
      pos,
      num_of_connections,
      ):
    super().__init__(model)
    self.agent_type = agent_type
    self.state_id = state_id
    self.num_of_connections = num_of_connections
    self.pos = pos

    self.adopted = False # Inital condition for all agents since the agent is seeded in the model
    self.is_seed = False # Inital condition for all agents since the agent is seeded in the model

  @classmethod
  def create_agents(cls, model, n, agent_type, num_of_connections, state_id, pos):
    """Factory method to create n agents."""
    agents = []
    for _ in range(n):
        agent = cls(model, agent_type, state_id, pos, num_of_connections)
        agents.append(agent)
    return agents

  def step(self):
    adopt_prb = 0.2
    state_agent_adopted = False
    if self.agent_type == "state": # if its a state agent
      state_agent_adopted = self.adopted
    else: # if its a local agent
      for s_agent in self.model.state_agents: # look for the parent state of the local agent
        if s_agent.state_id == self.state_id and s_agent.adopted: # if you find the state and the state has adopted it
          state_agent_adopted = True
          break
    if state_agent_adopted:
      adopt_prb += 0.2 # if the state has adopted the tech then the local agent in the state is more likely to adopt the innovation

    adopt_prb = min(1.0, adopt_prb) # Ensure the probability doesn't exceed 1
    # replace this with spreading code eventually
    #return print(f"Health Dept {self.unique_id} from State {self.state_id}")
    if not self.adopted:
      if random.random() < adopt_prb:
        self.adopted = True
        print(f"{self.unique_id} has adopted")
      else:
        print(f"{self.unique_id} has NOT adopted")


Health Model Class defines the structures of the model as well as the actions that can be taken in the model. The inputs to this model can be used as effectors to change simulation parameters and important outputs

In [249]:
class Health_Model(mesa.Model):
  def __init__(
      self,
      num_agents,
      num_of_connections,
      dist_threshold,
      diffusion_mode, # can also be geographical
      num_states_to_select,  # the number of states you wish to test in the mode;
      initial_seed, # = 1
      seed_includes_state, # while the state can be the seed, its random
      rng=4,
      state_data=None # New parameter to pass state_gdf
      ):
    super().__init__(rng=rng)

    self.num_agents = num_agents
    self.seed_includes_state = seed_includes_state
    self.num_of_connections = num_of_connections
    self.dist_threshold = dist_threshold
    self.num_states_to_select = num_states_to_select

    if state_data is None:
        raise ValueError("state_data (geopandas DataFrame) must be provided.")
    self.state_gdf = state_data
    self._build_state_adjacencies() # Build adjacencies once

    self.create_states() # This will now use num_states_to_select and selected borders
    self.create_local()

    self.choose_seed()
    self.network = nx.Graph()

    for agent in self.state_agents + self.local_agents: # Iterate over all agents for network creation
      self.network.add_node(agent.unique_id)
      self.network.nodes[agent.unique_id]['pos'] = agent.pos

    if diffusion_mode == "government":
      self.create_gov_network()
    else:
      self.create_local_network()


## Modify Here for data collection
    self.datacollector = mesa.DataCollector(
            model_reporters={"Adopters": lambda m: sum(a.adopted for a in m.agents)}
    )

  def _get_random_point_in_polygon(self, poly):
    minx, miny, maxx, maxy = poly.bounds
    while True:
      p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
      if poly.contains(p):
        return (p.x, p.y)

  def _build_state_adjacencies(self): # Finds what states border each other
    adj = {}
    for fips1, geom1 in self.state_gdf.geometry.items():
        adj[fips1] = []
        for fips2, geom2 in self.state_gdf.geometry.items():
            if fips1 != fips2 and geom1.touches(geom2):
                adj[fips1].append(fips2)
    self._state_adjacencies = adj

  def create_states(self):
    self.state_agents =[]
    selected_state_ids = []
    queue = []
    visited = set()

    # Find Kentucky's FIPS code and start with the best state (commonwealth)
    kentucky_fips_list = self.state_gdf[self.state_gdf['NAME'] == 'Kentucky'].index.tolist()
    if not kentucky_fips_list:
        raise ValueError("Kentucky not found in the state data. Please ensure 'All_States.geojson' contains 'Kentucky'.")
    kentucky_fips = kentucky_fips_list[0]


    # Start with Kentucky
    if kentucky_fips not in visited:
        selected_state_ids.append(kentucky_fips)
        queue.append(kentucky_fips)
        visited.add(kentucky_fips)

    # BFS to select 'num_states_to_select' states
    while queue and len(selected_state_ids) < self.num_states_to_select:
        current_state_fips = queue.pop(0) # BFS

        neighbors = self._state_adjacencies.get(current_state_fips, [])
        for neighbor_fips in neighbors:
            if neighbor_fips not in visited and len(selected_state_ids) < self.num_states_to_select:
                selected_state_ids.append(neighbor_fips)
                queue.append(neighbor_fips)
                visited.add(neighbor_fips)

    # Store the selected state IDs and their geometries for later use
    self.STATE_IDS_FOR_MODEL = selected_state_ids
    self.STATE_POLYGONS_FOR_MODEL = {sid: self.state_gdf.loc[sid].geometry for sid in self.STATE_IDS_FOR_MODEL}

    # Create agents for the selected states
    for sid in self.STATE_IDS_FOR_MODEL:
        poly = self.STATE_POLYGONS_FOR_MODEL[sid]
        cx, cy = poly.centroid.x, poly.centroid.y
        pos = (cx + random.uniform(-0.75,.75), cy+ random.uniform(-0.75,.75))
        #pos = self._get_random_point_in_polygon(poly)
        agents = Health_Agent.create_agents(model=self, n=1, agent_type="state", num_of_connections = 0,state_id = sid, pos=pos)
        self.state_agents.extend(agents)

  def create_local(self):
    self.local_agents =[]
    # Use self.STATE_IDS_FOR_MODEL here
    for i in range(self.num_agents - len(self.STATE_IDS_FOR_MODEL)): # desired agents - state agents = # of local agents
      sid = random.choice(self.STATE_IDS_FOR_MODEL)
      poly = self.state_gdf.loc[sid].geometry # Use the full state_gdf here, or self.STATE_POLYGONS_FOR_MODEL[sid]
      pos = self._get_random_point_in_polygon(poly)
      agents = Health_Agent.create_agents(model=self,n=1,agent_type="local", num_of_connections=0, state_id = sid, pos=pos)
      self.local_agents.extend(agents)

  def create_gov_network(self):
       # State → State (federal) Prioritized as first
    for i, s1 in enumerate(self.state_agents): # No quota on states!!!! SHould be connected to every state and every local in their district
        for s2 in self.state_agents[i+1:]: # Avoid self-loops and duplicate edges
            # Only add edge if they are adjacent based on the selected states
            # CHECK THIS LOGIC, maybe all should talk to every one??
            if s2.state_id in self._state_adjacencies.get(s1.state_id, []):
                self.network.add_edge(s1.unique_id, s2.unique_id)

       # State → Local Second Priority
    for s in self.state_agents:
      locals_in_state = [a for a in self.local_agents if a.state_id == s.state_id]
      for a in locals_in_state:
            self.network.add_edge(s.unique_id, a.unique_id)
            a.num_of_connections += 1

    # Local → Local (within same state) Last Priority
    # Use self.STATE_IDS_FOR_MODEL here
    for sid in self.STATE_IDS_FOR_MODEL:
        locals_in_state = [a for a in self.local_agents if a.state_id == sid]
        for i, a1 in enumerate(locals_in_state):
            for a2 in locals_in_state[i+1:]: # Avoid self-loops and duplicate edges
              if a1.num_of_connections < self.num_of_connections and a2.num_of_connections < self.num_of_connections:
                self.network.add_edge(a1.unique_id, a2.unique_id)
                a1.num_of_connections += 1
                a2.num_of_connections += 1

  def create_local_network(self):
        # State → State (federal) 1st priority
    for i, s1 in enumerate(self.state_agents):
        for s2 in self.state_agents[i+1:]: # Avoid self-loops and duplicate edges
            # Only add edge if they are adjacent based on the selected states
            if s2.state_id in self._state_adjacencies.get(s1.state_id, []):
                self.network.add_edge(s1.unique_id, s2.unique_id)

    # State → Local second priority
    for s in self.state_agents:
      locals_in_state = [a for a in self.local_agents if a.state_id == s.state_id]
      for a in locals_in_state:
            self.network.add_edge(s.unique_id, a.unique_id)
            a.num_of_connections += 1

    # Local to locals last priority
    for i, a1 in enumerate(self.local_agents):
      for a2 in self.local_agents[i+1:]: # Avoid self-loops and duplicate edges
          d = math.dist(a1.pos, a2.pos)
          if d <= self.dist_threshold:  # using this as distance threshold
              if a1.num_of_connections < self.num_of_connections and a2.num_of_connections < self.num_of_connections:
                    self.network.add_edge(a1.unique_id, a2.unique_id)
                    a1.num_of_connections += 1
                    a2.num_of_connections += 1

  def choose_seed(self):
    if self.seed_includes_state: # if the seed is a state agent
      seed = random.choice(self.state_agents)
    else:
      seed = random.choice(self.local_agents)
    seed.adopted = True
    seed.is_seed = True
    seed.exposures = 100 # random number of exposes since it doesn't matter
    self.seed_agent = seed

  def step(self):
    print(f"Health Dept {self.seed_agent.unique_id} from State {self.seed_agent.state_id} is the seed agent")
    self.agents.shuffle_do("step") # Let the schedule call step for all agents
    self.datacollector.collect(self)

Data Collection Module:

*  Roger's profile output analysis
* Adopters over Time



This will hold the graphs from the model running. Note that if you want to collect data as the model is running you should alter the Data collecter feature in the model definition. https://mesa.readthedocs.io/latest/apis/datacollection.html

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create widgets for model parameters
num_agents_slider = widgets.IntText(
    min=10, max=200, step=5, value=30,
    description='Total Agents:',
    continuous_update=False,
    style = {'description_width': 'initial'}
)

num_states_slider = widgets.IntText(
    min=2, max=state_gdf.shape[0] - len(states_to_remove), step=1, value=5,
    description='# of States To Simulate:',
    continuous_update=False,
    style = {'description_width': 'initial'}
)

num_connections_slider = widgets.IntSlider(
    min=1, max=10, step=1, value=3,
    description='Max Connections:',
    continuous_update=False,
    style = {'description_width': 'initial'}
)

dist_threshold_slider = widgets.IntSlider(
    min=1, max=10, step=1, value=3,
    description='Distance Threshold:',
    continuous_update=False,
     style = {'description_width': 'initial'}
)
diffusion_mode_dropdown = widgets.Dropdown(
    options=['government', 'local'],
    value='government',
    description='Diffusion Network:',
    style = {'description_width': 'initial'}
)

seed_includes_state_checkbox = widgets.Checkbox(
    value=False,
    description='Seed is State Agent',
    disabled=False,
    indent=False,
    style = {'description_width': 'initial'}
)

rng_text = widgets.IntText(
    value=4,
    description='Random Seed:',
    disabled=False,
    style = {'description_width': 'initial'}
)

# Create an output widget to display results

output = widgets.Output()

def run_model_and_plot(
    num_agents,
    num_states_to_select,
    num_of_connections,
    dist_threshold,
    diffusion_mode,
    seed_includes_state,
    rng
):
    with output:
        clear_output(wait=True)
        print("Running model with selected parameters...")
        try:
            model = Health_Model(
                num_agents=num_agents,
                num_states_to_select=num_states_to_select,
                initial_seed=1, # Initial seed is not user controlled in the model definition
                num_of_connections=num_of_connections,
                diffusion_mode=diffusion_mode,
                dist_threshold=dist_threshold,
                seed_includes_state=seed_includes_state,
                rng=rng,
                state_data=state_gdf
            )
            model.run_for(3)
            print(f"Names of selected states: {[state_gdf.loc[sid]['NAME'] for sid in model.STATE_IDS_FOR_MODEL]}")

            fig, ax = plt.subplots(figsize=(10, 10))

    # --- Determine state adoption status ---
            state_adoption_status = {}
            for sid in model.STATE_IDS_FOR_MODEL:
                agents_in_state = [a for a in model.state_agents + model.local_agents if a.state_id == sid]
                if agents_in_state:
                    adopted_count = sum(1 for a in agents_in_state if a.adopted)
                    majority_adopted = adopted_count > len(agents_in_state) / 2
                    state_adoption_status[sid] = majority_adopted
                else:
                    state_adoption_status[sid] = False # No agents, no adoption

            # --- Plot selected state polygons ---
            for sid, poly in model.STATE_POLYGONS_FOR_MODEL.items():
                current_state_color = 'green' if state_adoption_status.get(sid, False) else 'lightgray'
                if isinstance(poly, MultiPolygon):
                    for p in poly.geoms:
                        xs, ys = p.exterior.xy
                        ax.fill(xs, ys, alpha=0.5, fc=current_state_color, ec='black')
                else:
                    xs, ys = poly.exterior.xy
                    ax.fill(xs, ys, alpha=0.5, fc=current_state_color, ec='black')


            # --- Plot network ---
            positions = nx.get_node_attributes(model.network, 'pos')

            # Draw edges
            nx.draw_networkx_edges(model.network, positions, ax=ax, edge_color='gray')
            node_colors = []
            node_list = []
            for agent in model.state_agents + model.local_agents:
                node_list.append(agent.unique_id)
                if agent.is_seed:
                    node_colors.append('gold') # Seed agent
                elif agent.adopted:
                    node_colors.append('green') # Adopted agents
                else:
                    node_colors.append('red') # Non-adopted agents

            state_nodes_for_shape = [a.unique_id for a in model.state_agents]
            local_nodes_for_shape = [a.unique_id for a in model.local_agents]

            nx.draw_networkx_nodes(
                model.network,
                positions,
                nodelist=state_nodes_for_shape,
                node_shape='s', # Square for state agents
                node_size=100,
                node_color=[node_colors[node_list.index(n)] for n in state_nodes_for_shape], # Maintain adoption color
                ax=ax
            )

            nx.draw_networkx_nodes(
                model.network,
                positions,
                nodelist=local_nodes_for_shape,
                node_shape='o', # Circle for local agents
                node_size=50,
                node_color=[node_colors[node_list.index(n)] for n in local_nodes_for_shape], # Maintain adoption color
                ax=ax
            )

            # Draw seed agent
            nx.draw_networkx_nodes(
                model.network,
                positions,
                nodelist=[model.seed_agent.unique_id],
                node_color='gold',
                node_size=50,
                ax=ax
            )

            plt.title("Healthcare Innovation Network ")
            plt.axis("equal")
            plt.show()

            # Data collection and plotting for adopters over time
            agents_data = model.datacollector.get_model_vars_dataframe()
            plt.figure(figsize=(8, 8))
            a = sns.lineplot(data=agents_data)
            a.set(title="Adopters Over Model Steps", xlabel= "Model Steps", ylabel="Number of Adopters")
            plt.show()

        except Exception as e:
            print(f"An error occurred: {e}")

# Use interactive_output to link widgets to the function
interactive_plot = widgets.interactive_output(
    run_model_and_plot,
    {
        'num_agents': num_agents_slider,
        'num_states_to_select': num_states_slider,
        'num_of_connections': num_connections_slider,
        'dist_threshold': dist_threshold_slider,
        'diffusion_mode': diffusion_mode_dropdown,
        'seed_includes_state': seed_includes_state_checkbox,
        'rng': rng_text
    }
)

# Arrange and display the widgets and output
display(
    widgets.VBox([
        widgets.HBox([
            num_agents_slider,
        ], layout=widgets.Layout(width='auto', padding='10px')),
        widgets.HBox([
             num_connections_slider
        ], layout=widgets.Layout(width='auto', padding='10px')),
        widgets.HBox([
           num_states_slider,
        ], layout=widgets.Layout(width='auto', padding='10px')),
        widgets.HBox([
            dist_threshold_slider
        ], layout=widgets.Layout(width='auto', padding='10px')),
           widgets.HBox([
            diffusion_mode_dropdown, seed_includes_state_checkbox
        ], layout=widgets.Layout(width='auto', padding='10px')),
        rng_text,
        output
    ], layout=widgets.Layout(border='2px solid lightgray', padding='10px', margin='10px'))
)
